# GSOM Idionomic Clustering — Colab

Build and cluster a **Growing Self-Organizing Map** of person-specific feature matrices.

Powered by [`gsom-idionomic`](https://github.com/ozziejoe/gsom-idionomic) — the same engine as the web app.

Run each cell top to bottom (`Shift+Enter`); adjust the form fields on the right.

## 1. Install

In [ ]:
!pip install -q git+https://github.com/ozziejoe/gsom-idionomic.git
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
from gsom_idionomic import GsomConfig, build_map, cluster_map, make_sample_dataset
from gsom_idionomic import plots
from gsom_idionomic.demo import sample_legend, SAMPLE_RECOMMENDED
print('Ready.')

## 2. Load data
Use the built-in **sample** (80 people, four interpretable person-types) or upload your own
feature-matrix CSV (`ID` + `beta_*` columns, optional `se_*`).

In [ ]:
USE_SAMPLE = True  #@param {type:"boolean"}
ID_COL = 'ID'      #@param {type:"string"}

if USE_SAMPLE:
    df = make_sample_dataset()
    print('Sample data:', df.shape, '| four types:', sample_legend())
else:
    from google.colab import files
    up = files.upload(); df = pd.read_csv(list(up.keys())[0])
    print('Loaded:', df.shape)
df.head()

## 3. Step 1 — Build & tune the map
For the sample, the defaults below (spread 0.6) recover the four types cleanly.
For your own data, try `SPREAD_MODE = 'auto'`.

In [ ]:
SPREAD_MODE = 'hand'  #@param ["auto", "default_0.80", "hand"]
HAND_SPREAD = 0.6     #@param {type:"slider", min:0.3, max:0.95, step:0.05}
TRAINING_ITER = 100   #@param {type:"slider", min:20, max:200, step:10}
SMOOTHING_ITER = 50   #@param {type:"slider", min:10, max:100, step:10}

cfg = GsomConfig(id_col=ID_COL, training_iter=TRAINING_ITER, smoothing_iter=SMOOTHING_ITER)
spread = {'auto':'auto','default_0.80':0.80,'hand':HAND_SPREAD}[SPREAD_MODE]
m = build_map(df, cfg, spread=spread)
print(f'spread={m.spread} | active nodes={m.n_nodes} | winner nodes={m.n_winner_nodes}')
if m.df_spread is not None: display(plots.plot_spread_tuning(m.df_spread, m.spread))

## 4. Step 2 — Cluster the map
For the sample, `K = 4` with outlier cutoff `0.10` gives the clean recovery.

In [ ]:
K_MODE = 'hand'       #@param ["auto", "hand"]
HAND_K = 4            #@param {type:"slider", min:2, max:12, step:1}
OUTLIER_CUTOFF = 0.10 #@param {type:"slider", min:0.0, max:0.6, step:0.05}

cfg.sil_cut = OUTLIER_CUTOFF
k = 'auto' if K_MODE=='auto' else HAND_K
c = cluster_map(m, cfg, k=k)
print(f'K={c.best_k} | clusters={c.valid_clusters} | outliers={(c.df_clusters.cluster==-1).sum()}')

display(plots.plot_skeleton_map(m.gsom_map, c.df_active, m.df_map, c.df_profiles, ID_COL))
display(plots.plot_cluster_map(c.df_active, m.df_map))
display(plots.plot_silhouette(c.df_active, cfg.sil_cut))
for cl in c.valid_clusters: display(plots.plot_divergence(cl, c, cfg))
if any(col.startswith('se_') for col in df.columns): display(plots.plot_homogeneity_gain(c, cfg))

## 5. Recovery check (sample only)
Designed type × GSOM cluster — a clean block-diagonal means perfect recovery.

In [ ]:
if USE_SAMPLE:
    xt = c.df_clusters.copy(); xt['Designed type'] = xt['ID'].str[:3].map(sample_legend())
    display(pd.crosstab(xt['Designed type'], xt['cluster']))

## 6. Tables

In [ ]:
print('Summary (journal) table:'); display(c.df_summary_table)
print('2x2 classification:'); display(c.df_classification.head(20))

## 7. Download results

In [ ]:
import zipfile, os
os.makedirs('gsom_output/tables', exist_ok=True); os.makedirs('gsom_output/figures', exist_ok=True)
c.df_clusters.to_csv('gsom_output/tables/cluster_assignments.csv', index=False)
c.df_characteristics.to_csv('gsom_output/tables/characteristics.csv', index=False)
c.df_classification.to_csv('gsom_output/tables/classification_2x2.csv', index=False)
c.df_full_table.to_csv('gsom_output/tables/cluster_table_full.csv', index=False)
c.df_summary_table.to_csv('gsom_output/tables/cluster_table_summary.csv', index=False)
plots.plot_skeleton_map(m.gsom_map, c.df_active, m.df_map, c.df_profiles, ID_COL).savefig('gsom_output/figures/skeleton_map.png', dpi=200, bbox_inches='tight')
plots.plot_cluster_map(c.df_active, m.df_map).savefig('gsom_output/figures/cluster_map.png', dpi=200, bbox_inches='tight')
plots.plot_silhouette(c.df_active, cfg.sil_cut).savefig('gsom_output/figures/silhouette.png', dpi=200, bbox_inches='tight')
with zipfile.ZipFile('gsom_results.zip','w',zipfile.ZIP_DEFLATED) as zf:
    for root,_,fs in os.walk('gsom_output'):
        for f in fs: zf.write(os.path.join(root,f))
from google.colab import files; files.download('gsom_results.zip')